# Load and Re-organize Data

In [ ]:
from pathlib import Path

BASE_PATH = Path('./datasets/PIQA')

train_data_path = BASE_PATH / 'train.jsonl'
train_label_path = BASE_PATH / 'train-labels.lst'

test_data_path = BASE_PATH / 'test.jsonl'
test_label_path = BASE_PATH / 'test-labels.lst'

In [28]:
import pandas as pd

train_dataset = pd.read_json(train_data_path, lines=True, encoding="utf-8")
train_labels = pd.read_csv(train_label_path, header=None, names=["label"])

train_dataset = pd.concat([train_dataset.reset_index(drop=True), train_labels], axis=1)

test_dataset = pd.read_json(test_data_path, lines=True, encoding="utf-8")
test_labels = pd.read_csv(test_label_path, header=None, names=["label"])

test_dataset = pd.concat([test_dataset.reset_index(drop=True), test_labels], axis=1)

print(train_dataset.shape)
print(test_dataset.shape)

(16113, 4)
(1838, 4)


# Train/Validate/Test Data Split

In [ ]:
proportion = 0.15
sample_size = int(len(train_dataset) * proportion)

valid_dataset = train_dataset.sample(n=sample_size, random_state=4012)
train_dataset = train_dataset.drop(valid_dataset.index)

In [31]:
print(train_dataset.shape)
print(valid_dataset.shape)
print(test_dataset.shape)

(14502, 4)
(1611, 4)
(1838, 4)


In [27]:
train_dataset.head()

,goal,sol1,sol2,label
0,"When boiling butter, when it's ready, you can",Pour it onto a plate,Pour it into a jar,1
2,how do you indent something?,leave a space before starting the writing,press the spacebar,0
4,Clean tires,"Pour water, cape off caked on dirt. Use speed...","Pour water, scrape off caked on dirt. Use a st...",1
5,how do you taste something?,smell it enough to taste it.,place it in your mouth to taste.,1
6,"To create a makeshift ice pack,",take a sponge and soak it in oil. Put the spon...,take a sponge and soak it in water. Put the sp...,1


# Train Tokenizer

In [41]:
# Prepare temporary training corpus
import random

temp_corpus = []
temp_corpus.extend(train_dataset['goal'].to_list())
temp_corpus.extend(train_dataset['sol1'].to_list())
temp_corpus.extend(train_dataset['sol2'].to_list())

random.Random(42).shuffle(temp_corpus)

temp_corpus[0:3]

['Place a piece of tape around the area you are planning to punch',
 'can fill a  shot glass ',
 'Make triangle toy holder.']

In [42]:
output_path = Path("temp_corpus.txt")

with output_path.open("w", encoding="utf-8") as f:
    for text in temp_corpus:
        # clean potential returns inside corpus
        text = str(text).replace("\r\n", " ").replace("\n", " ").replace("\r", " ")
        f.write(text + "\n")

In [57]:
# Train Tokenizer

import sentencepiece as spm

spm.SentencePieceTrainer.train(input=str(output_path),
                                model_prefix="piqa_bpe",
                                model_type="bpe",

                                vocab_size=8000,
                                character_coverage=1.0,
                                byte_fallback=True,

                                unk_id=0,
                                bos_id=1,
                                eos_id=2,
                                pad_id=3,

                                user_defined_symbols=["[CLS]", "[SEP]"]
                            )

True

In [60]:
# Load tokenizer and test

import sentencepiece as spm

tokenizer = spm.SentencePieceProcessor(
    model_file="piqa_bpe.model"
)

text = "An unseen-word, café, or 😊 is still tokenizable."

pieces = tokenizer.encode(text, out_type=str)
ids = tokenizer.encode(text, out_type=int)

print(pieces)
print(ids)

# restore sentence
restored = tokenizer.decode(ids)
print(restored)

['▁An', '▁un', 'se', 'en', '-', 'word', ',', '▁c', 'af', 'é', ',', '▁or', '▁', '<0xF0>', '<0x9F>', '<0x98>', '<0x8A>', '▁is', '▁still', '▁to', 'k', 'en', 'iz', 'able', '.']
[3460, 779, 319, 299, 7932, 6122, 7915, 269, 1390, 7976, 7915, 366, 7892, 246, 165, 158, 144, 379, 1726, 278, 7914, 299, 868, 774, 7913]
An unseen-word, café, or 😊 is still tokenizable.


In [47]:
vocab_size = tokenizer.get_piece_size()
vocab_size

8000

In [ ]:
print("UNK:", tokenizer.unk_id())  # 0
print("BOS:", tokenizer.bos_id())  # 1
print("EOS:", tokenizer.eos_id())  # 2
print("PAD:", tokenizer.pad_id())  # 3

UNK: 0
BOS: 1
EOS: 2
PAD: 3


In [62]:
cls_id = tokenizer.piece_to_id("[CLS]")
sep_id = tokenizer.piece_to_id("[SEP]")

print(cls_id, sep_id)

4 5
